# 🎵 Audio Properties Tutorial
### Amplitude, Frequency, Wavelength, Phase
Tutorial lengkap dari nol — jalankan dari atas ke bawah tanpa error.

## 📦 Cell 1 — Import & Global Config
Semua import dan konstanta global didefinisikan di sini.

In [ ]:
import torch
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ─── Global config ────────────────────────────────────────────
SR         = 16000   # sample rate (Hz)
DURATION   = 2.0     # durasi sinyal (detik)
FREQ_HZ    = 440     # frekuensi nada A4 (Hz)
N_FFT      = 1024    # jumlah titik FFT
N_SAMPLES  = int(SR * DURATION)

# ─── Time axis ────────────────────────────────────────────────
t = torch.linspace(0, DURATION, N_SAMPLES)  # [0 .. 2.0] detik

print(f"Sample rate  : {SR} Hz")
print(f"Duration     : {DURATION} s")
print(f"Total samples: {N_SAMPLES}")
print(f"Time axis    : shape={t.shape}, min={t.min():.3f}, max={t.max():.3f}")

## 🎛️ Cell 2 — Generate Sinyal Sintetis
Kita buat sinyal sederhana dulu agar bisa dikontrol penuh.
Nanti di Cell 3 kamu bisa ganti dengan file audio asli.

In [ ]:
# Sinyal pure sine wave pada frekuensi FREQ_HZ
waveform_synth = torch.sin(2 * torch.pi * FREQ_HZ * t)  # shape: [N_SAMPLES]

# torchaudio convention: [channels, samples]
waveform_synth = waveform_synth.unsqueeze(0)  # shape: [1, N_SAMPLES]

print(f"Waveform shape : {waveform_synth.shape}")
print(f"Channels       : {waveform_synth.shape[0]}")
print(f"Samples        : {waveform_synth.shape[1]}")

# Gunakan channel pertama untuk analisis
signal = waveform_synth[0]  # shape: [N_SAMPLES]

plt.figure(figsize=(12, 3))
plt.plot(t[:500].numpy(), signal[:500].numpy(), color='steelblue')
plt.title(f"Synthetic Sine Wave — {FREQ_HZ} Hz")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

## 📂 Cell 3 — (Opsional) Load Audio dari File
Kalau punya file audio sendiri, jalankan cell ini.
Kalau tidak, skip — variabel `signal` dan `t` sudah tersedia dari Cell 2.

In [ ]:
# ── Ganti path di bawah ini dengan file audio kamu ──
AUDIO_PATH = "dataset\\cats_dogs\\cats_dogs\\cat_1.wav"

# Uncomment block ini kalau mau pakai file asli:
waveform_file, sr_file = torchaudio.load(AUDIO_PATH)

# Resample ke SR global jika berbeda
if sr_file != SR:
    resampler = T.Resample(orig_freq=sr_file, new_freq=SR)
    waveform_file = resampler(waveform_file)

# Ambil channel pertama, hitung ulang t & signal
signal    = waveform_file[0]
N_SAMPLES = signal.shape[0]
DURATION  = N_SAMPLES / SR
t         = torch.linspace(0, DURATION, N_SAMPLES)

print(f"File loaded  : {AUDIO_PATH}")
print(f"Duration     : {DURATION:.2f} s")
print(f"Total samples: {N_SAMPLES}")

# print("Cell 3 skipped — menggunakan sinyal sintetis dari Cell 2.")

## 🔊 Cell 4 — Amplitude
Seberapa **keras/kuat** sinyal — nilai absolut dari tiap sample.

In [ ]:
# ── Statistik Amplitude ─────────────────────────────────────
amp_max  = signal.max().item()
amp_min  = signal.min().item()
amp_mean = signal.abs().mean().item()
amp_rms  = torch.sqrt(torch.mean(signal ** 2)).item()
amp_db   = 20 * np.log10(amp_rms + 1e-9)

print("─── Amplitude Statistics ───")
print(f"Max amplitude  : {amp_max:.4f}")
print(f"Min amplitude  : {amp_min:.4f}")
print(f"Mean |amplitude|: {amp_mean:.4f}")
print(f"RMS (loudness) : {amp_rms:.4f}")
print(f"Loudness (dB)  : {amp_db:.2f} dB")

# ── Visualisasi ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
axes[0].plot(t[:800].numpy(), signal[:800].numpy(), color='steelblue', linewidth=0.8)
axes[0].axhline(amp_rms,  color='red',    linestyle='--', label=f'RMS = {amp_rms:.2f}')
axes[0].axhline(-amp_rms, color='red',    linestyle='--')
axes[0].axhline(amp_max,  color='orange', linestyle=':',  label=f'Max = {amp_max:.2f}')
axes[0].set_title("Waveform & Amplitude")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].legend()
axes[0].grid(True)

# Envelope (amplitudo absolut)
axes[1].plot(t.numpy(), signal.abs().numpy(), color='coral', linewidth=0.6)
axes[1].axhline(amp_rms, color='red', linestyle='--', label=f'RMS = {amp_rms:.2f}')
axes[1].set_title("Amplitude Envelope |signal|")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("|Amplitude|")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 🎵 Cell 5 — Frequency & Spectrum (FFT)
Seberapa **cepat** gelombang berulang — menentukan nada suara.

In [ ]:
# ── FFT ─────────────────────────────────────────────────────
fft_result  = torch.fft.rfft(signal, n=N_FFT)
magnitude   = torch.abs(fft_result)
freqs       = torch.fft.rfftfreq(N_FFT, d=1.0/SR)  # frekuensi axis (Hz)

# Dominant frequency
dominant_idx  = torch.argmax(magnitude).item()
dominant_freq = freqs[dominant_idx].item()

print("─── Frequency Analysis ───")
print(f"FFT points (N_FFT)   : {N_FFT}")
print(f"Freq resolution      : {SR / N_FFT:.2f} Hz per bin")
print(f"Freq range           : 0 – {SR/2:.0f} Hz (Nyquist)")
print(f"Dominant frequency   : {dominant_freq:.1f} Hz")
print(f"Expected frequency   : {FREQ_HZ} Hz")

# Top 5 frequencies
top5_idx   = torch.topk(magnitude, 5).indices
print("\nTop 5 frequencies:")
for i, idx in enumerate(top5_idx):
    print(f"  #{i+1}: {freqs[idx].item():.1f} Hz  (magnitude={magnitude[idx].item():.1f})")

# ── Visualisasi ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full spectrum
axes[0].plot(freqs.numpy(), magnitude.numpy(), color='steelblue', linewidth=0.8)
axes[0].axvline(dominant_freq, color='red', linestyle='--', label=f'Dominant: {dominant_freq:.1f} Hz')
axes[0].set_title("Frequency Spectrum (Full)")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Magnitude")
axes[0].legend()
axes[0].grid(True)

# Zoomed spectrum (0–2000 Hz)
zoom_mask = freqs <= 2000
axes[1].plot(freqs[zoom_mask].numpy(), magnitude[zoom_mask].numpy(), color='coral', linewidth=0.8)
axes[1].axvline(dominant_freq, color='red', linestyle='--', label=f'Dominant: {dominant_freq:.1f} Hz')
axes[1].set_title("Frequency Spectrum (Zoom 0–2000 Hz)")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Magnitude")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 📏 Cell 6 — Wavelength & Period
**Panjang satu siklus** gelombang — dihitung dari frekuensi.

In [ ]:
SPEED_OF_SOUND = 343.0  # m/s di udara suhu ruang

# Hitung wavelength dari dominant frequency
wavelength        = SPEED_OF_SOUND / dominant_freq
period_sec        = 1.0 / dominant_freq
samples_per_cycle = SR / dominant_freq

print("─── Wavelength & Period ───")
print(f"Dominant frequency  : {dominant_freq:.1f} Hz")
print(f"Period (T)          : {period_sec*1000:.4f} ms  (= 1/f)")
print(f"Wavelength (λ)      : {wavelength:.4f} m   (= v/f, v={SPEED_OF_SOUND} m/s)")
print(f"Samples per cycle   : {samples_per_cycle:.1f} samples")

# Perbandingan beberapa nada
notes = {
    "C4 (261 Hz)" : 261.63,
    "A4 (440 Hz)" : 440.00,
    "C5 (523 Hz)" : 523.25,
    "A5 (880 Hz)" : 880.00,
}
print("\n─── Wavelength Comparison ───")
print(f"{'Note':<15} {'Freq (Hz)':<12} {'Period (ms)':<14} {'Wavelength (m)':<16} {'Samples/cycle'}")
print("-" * 70)
for name, f in notes.items():
    wl  = SPEED_OF_SOUND / f
    per = 1000 / f
    spc = SR / f
    print(f"{name:<15} {f:<12.2f} {per:<14.3f} {wl:<16.4f} {spc:.1f}")

# ── Visualisasi satu siklus ────────────────────────────────
one_cycle_samples = int(samples_per_cycle)
t_one_cycle = t[:one_cycle_samples * 3]  # tampilkan 3 siklus
sig_one_cycle = signal[:one_cycle_samples * 3]

plt.figure(figsize=(12, 3))
plt.plot(t_one_cycle.numpy(), sig_one_cycle.numpy(), color='steelblue')
for i in range(1, 4):
    x_mark = period_sec * i
    plt.axvline(x_mark, color='red', linestyle='--', alpha=0.6,
                label='Period' if i == 1 else None)
plt.title(f"3 Siklus — Period = {period_sec*1000:.2f} ms | Wavelength = {wavelength:.4f} m")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 🔄 Cell 7 — Phase
**Posisi awal** gelombang — seberapa 'geser' sinyal terhadap referensi.

In [ ]:
# ── Phase dari FFT ──────────────────────────────────────────
fft_result = torch.fft.rfft(signal, n=N_FFT)
phase      = torch.angle(fft_result)  # radian, range [-π, π]

print("─── Phase Analysis ───")
print(f"Phase range : [{phase.min():.3f}, {phase.max():.3f}] radian")
print(f"Phase range : [{torch.rad2deg(phase.min()):.1f}, {torch.rad2deg(phase.max()):.1f}] derajat")

# Phase pada dominant frequency
dominant_idx   = torch.argmax(torch.abs(fft_result)).item()
dominant_phase = phase[dominant_idx].item()
print(f"\nPhase pada freq dominan ({dominant_freq:.1f} Hz):")
print(f"  {dominant_phase:.4f} radian = {np.rad2deg(dominant_phase):.2f} derajat")

# ── Buat beberapa sinyal dengan phase berbeda ────────────────
shifts = {
    "0°   (original)" : 0,
    "90°  (π/2)"      : torch.pi / 2,
    "180° (π)"        : torch.pi,
    "270° (3π/2)"     : 3 * torch.pi / 2,
}

# Tampilkan 3 siklus
n_show = int(SR / dominant_freq * 3)
t_show = t[:n_show]

# ── Visualisasi phase shift ──────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

colors = ['steelblue', 'coral', 'green', 'purple']
for (label, shift), color in zip(shifts.items(), colors):
    s = torch.sin(2 * torch.pi * dominant_freq * t_show + shift)
    axes[0].plot(t_show.numpy(), s.numpy(), label=label, color=color, linewidth=1.5)

axes[0].set_title("Phase Shift Comparison")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].legend(loc='upper right')
axes[0].grid(True)

# Phase spectrum
zoom_mask = freqs <= 2000
axes[1].plot(freqs[zoom_mask].numpy(),
             torch.rad2deg(phase[zoom_mask]).numpy(),
             color='steelblue', linewidth=0.7)
axes[1].axvline(dominant_freq, color='red', linestyle='--',
                label=f'Dominant: {dominant_freq:.1f} Hz')
axes[1].set_title("Phase Spectrum (0–2000 Hz)")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Phase (degrees)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 📊 Cell 8 — Dashboard Lengkap (Semua Properties)
Visualisasi keempat properties sekaligus dalam satu figure.

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

n_show = int(SR / dominant_freq * 4)  # 4 siklus
t_show = t[:n_show]
s_show = signal[:n_show]

# ── 1. Waveform & Amplitude ───────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t_show.numpy(), s_show.numpy(), color='steelblue', linewidth=1.2)
ax1.axhline( amp_rms, color='red',    linestyle='--', linewidth=1, label=f'RMS = {amp_rms:.3f}')
ax1.axhline(-amp_rms, color='red',    linestyle='--', linewidth=1)
ax1.axhline( amp_max, color='orange', linestyle=':',  linewidth=1, label=f'Max = {amp_max:.3f}')
ax1.set_title(f"① Waveform & Amplitude  |  RMS={amp_rms:.3f}, Max={amp_max:.3f}, {amp_db:.1f} dB",
              fontsize=11)
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Amplitude")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.4)

# ── 2. Frequency Spectrum ────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
zoom_mask = freqs <= 2000
ax2.plot(freqs[zoom_mask].numpy(), magnitude[zoom_mask].numpy(), color='coral', linewidth=0.9)
ax2.axvline(dominant_freq, color='red', linestyle='--', linewidth=1.2,
            label=f'Peak: {dominant_freq:.1f} Hz')
ax2.set_title(f"② Frequency Spectrum  |  Dominant={dominant_freq:.1f} Hz", fontsize=11)
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Magnitude")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4)

# ── 3. Wavelength — 1 siklus ─────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
one_cycle = int(samples_per_cycle)
t_cyc     = t[:one_cycle]
s_cyc     = signal[:one_cycle]
ax3.plot(t_cyc.numpy(), s_cyc.numpy(), color='green', linewidth=1.5)
ax3.axvspan(0, period_sec, alpha=0.15, color='green', label=f'λ period = {period_sec*1000:.2f} ms')
ax3.set_title(f"③ Wavelength / Period  |  T={period_sec*1000:.2f} ms, λ={wavelength:.3f} m",
              fontsize=11)
ax3.set_xlabel("Time (s)")
ax3.set_ylabel("Amplitude")
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.4)

# ── 4. Phase Shift ────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, :])
n_ps   = int(SR / dominant_freq * 3)
t_ps   = t[:n_ps]
colors = ['steelblue', 'coral', 'green', 'purple']
labels = ["0° original", "90° (π/2)", "180° (π)", "270° (3π/2)"]
pshifts = [0, torch.pi/2, torch.pi, 3*torch.pi/2]
for ph, lbl, col in zip(pshifts, labels, colors):
    s_ph = torch.sin(2 * torch.pi * dominant_freq * t_ps + ph)
    ax4.plot(t_ps.numpy(), s_ph.numpy(), label=lbl, color=col,
             linewidth=1.4, alpha=0.85)
ax4.set_title("④ Phase Shift Comparison", fontsize=11)
ax4.set_xlabel("Time (s)")
ax4.set_ylabel("Amplitude")
ax4.legend(loc='upper right', fontsize=9, ncol=4)
ax4.grid(True, alpha=0.4)

fig.suptitle(f"Audio Properties Dashboard  —  {dominant_freq:.0f} Hz, SR={SR} Hz",
             fontsize=13, fontweight='bold')
plt.show()

## 📝 Cell 9 — Ringkasan & Relasi Antar Properties

In [ ]:
print("=" * 55)
print(" RINGKASAN AUDIO PROPERTIES")
print("=" * 55)
print(f"""
① AMPLITUDE
   • Max      : {amp_max:.4f}
   • RMS      : {amp_rms:.4f}
   • Loudness : {amp_db:.2f} dB
   → Semakin besar amplitudo, semakin keras suara.

② FREQUENCY
   • Dominant : {dominant_freq:.1f} Hz
   • Range    : 0 – {SR//2} Hz (Nyquist = SR/2)
   → Semakin tinggi frekuensi, semakin tinggi nada.

③ WAVELENGTH / PERIOD
   • Period   : {period_sec*1000:.4f} ms  (T = 1/f)
   • Wavelength: {wavelength:.4f} m  (λ = v/f, v=343 m/s)
   • Samples/cycle: {samples_per_cycle:.1f}
   → Frekuensi tinggi = periode pendek = wavelength pendek.

④ PHASE
   • Phase dominan: {dominant_phase:.4f} rad = {np.rad2deg(dominant_phase):.2f}°
   • Range        : [-π, π] radian / [-180°, 180°]
   → Phase menentukan 'titik mulai' gelombang.
     Dua sinyal frekuensi sama tapi phase beda
     bisa saling memperkuat (konstruktif) atau
     melemahkan (destruktif/cancel).

RELASI:
   f (Hz)  = 1 / T
   T (s)   = 1 / f
   λ (m)   = v / f   (v = kecepatan suara)
   dB      = 20 * log10(RMS)
""")
print("=" * 55)
print(" NEXT STEP: Spectrogram & MFCC")
print("=" * 55)

## 🕹️ Cell 10 — Interactive Playback & Amplitude Visualizer
Gunakan slider untuk melihat amplitudo pada waktu tertentu, atau klik 'Play' untuk melihat animasi sinkronisasi.

In [ ]:
import librosa
import librosa.display
import IPython.display as ipd
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, Output
import time
import matplotlib.pyplot as plt

# ─── Interactive Setup ───
out = Output()

def plot_interactive(time_s=0.0):
    with out:
        clear_output(wait=True)
        # Ensure we use a consistent plot size
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # Plot Waveform menggunakan librosa (lebih premium)
        librosa.display.waveshow(signal.numpy(), sr=SR, alpha=0.5, color='steelblue', label='Waveform', ax=ax)
        
        # Indikator Garis Merah
        ax.axvline(x=time_s, color='red', linestyle='--', linewidth=2, label=f'Time: {time_s:.2f}s')
        
        ax.set_title("Interactive Amplitude Visualizer", fontsize=12, fontweight='bold', pad=20)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Amplitude")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# Slider
t_slider = FloatSlider(value=0.0, min=0.0, max=float(DURATION), step=0.01,
                       description='Seek Time:', layout={'width': '80%'})

# Output sinkronisasi slider
interactive_plot = widgets.interactive_output(plot_interactive, {'time_s': t_slider})

# Audio Widget
audio_player = ipd.Audio(signal.numpy(), rate=SR)

# Play with animation
btn_play = Button(description="Play with Cursor", icon="play", button_style='primary', layout={'width': '15%'})

def on_play_click(b):
    start_time = time.time()
    while time.time() - start_time < float(DURATION):
        elapsed = time.time() - start_time
        t_slider.value = min(elapsed, float(DURATION))
        time.sleep(0.05)
    t_slider.value = float(DURATION)

btn_play.on_click(on_play_click)

# Dashboard Layout
dashboard = VBox([
    HBox([btn_play, t_slider]),
    out,
    audio_player
])

# Explicitly display the dashboard
display(dashboard)

# Initial plot call to force first render
plot_interactive(0.0)
